In [5]:
import os

bronze_dir = 'data_lake/bronze'
silver_dir = 'data_lake/silver/preprocessed_sales'

os.makedirs(bronze_dir, exist_ok=True)
os.makedirs(silver_dir, exist_ok=True)

print("Folder structure created successfully:")
for root, dirs, files in os.walk('data_lake'):
    level = root.replace('data_lake', '').count(os.sep)
    indent = ' ' * 4 * (level)
    print(f"{indent}{os.path.basename(root)}/")

Folder structure created successfully:
data_lake/
    bronze/
    silver/
        preprocessed_sales/


In [7]:
from google.colab import files
import shutil
import os

print("Please upload your 'sales.csv' file below:")
uploaded = files.upload()

for filename in uploaded.keys():
    target_path = os.path.join('data_lake/bronze', filename)
    shutil.move(filename, target_path)
    print(f"\nSuccessfully uploaded and moved '{filename}' to '{target_path}'")

Please upload your 'sales.csv' file below:


Saving sales.csv to sales.csv

Successfully uploaded and moved 'sales.csv' to 'data_lake/bronze/sales.csv'


In [8]:
!pip install pyspark

In [9]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F

spark = SparkSession.builder \
    .appName("DataLakeBronzeExploration") \
    .getOrCreate()

bronze_path = "data_lake/bronze/sales.csv"
df = spark.read.csv(bronze_path, header=True, inferSchema=True)

print("=== DataFrame Schema ===")
df.printSchema()

print(f"Total Record Count: {df.count()}")

print("\n=== Sample Records ===")
df.show(5, truncate=False)

=== DataFrame Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)

Total Record Count: 990

=== Sample Records ===
+--------+-----------+------------------+----------------------+---------+--------+----------+----------------+----------------+----------+-----------+----------+------------+
|order_id|customer_id|customer_name     |product               |category |quantity|unit_price|discount_percent|payment_method  |city      |state      |order_date|order_status|
+--------+-----------+---

In [10]:
# 1. Identify Duplicates
duplicate_count = df.count() - df.dropDuplicates().count()
print(f"Number of exact duplicate rows: {duplicate_count}")

Number of exact duplicate rows: 0


In [11]:
# 2. Identify Null Values per Column
print("=== Null Values Count per Column ===")
from pyspark.sql import functions as F

null_counts = [
    F.sum(
        F.when(
            F.col(c).isNull() | (F.col(c).cast("string") == ""),
            1
        ).otherwise(0)
    ).alias(c)
    for c in df.columns
]

df.select(null_counts).show()

=== Null Values Count per Column ===
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|         6|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+



In [12]:
# 3. Analyze Text Format Inconsistencies & Numerical Anomalies
from pyspark.sql import functions as F

print("=== Unique Category Values ===")
df.groupBy("category").count().show(truncate=False)

print("=== Numerical Anomalies ===")
num_cols = [c for c, t in df.dtypes if t in ["int", "bigint", "float", "double"]]

for c in num_cols:
    print(f"\nStats for {c}:")
    df.select(
        F.min(c).alias("min"),
        F.max(c).alias("max"),
        F.sum(F.when(F.col(c) < 0, 1).otherwise(0)).alias("negative_count")
    ).show()

=== Unique Category Values ===
+---------------+-----+
|category       |count|
+---------------+-----+
|Sports         |138  |
|Grocery        |134  |
|Electronics    |104  |
|Clothing       |108  |
|Books          |127  |
|Home Appliances|113  |
|Furniture      |129  |
|Beauty         |137  |
+---------------+-----+

=== Numerical Anomalies ===

Stats for order_id:
+------+------+--------------+
|   min|   max|negative_count|
+------+------+--------------+
|100001|101000|             0|
+------+------+--------------+


Stats for quantity:
+---+---+--------------+
|min|max|negative_count|
+---+---+--------------+
|  1| 10|             0|
+---+---+--------------+


Stats for unit_price:
+---+-----+--------------+
|min|  max|negative_count|
+---+-----+--------------+
|  0|81010|             0|
+---+-----+--------------+


Stats for discount_percent:
+---+---+--------------+
|min|max|negative_count|
+---+---+--------------+
|  0| 40|             0|
+---+---+--------------+



In [13]:
date_cols = [c for c in df.columns if "date" in c.lower()]

for c in date_cols:
    print(f"\n=== Checking {c} (Correct format: yyyy-MM-dd) ===")
    parsed = F.try_to_timestamp(F.col(c), F.lit("yyyy-MM-dd"))

    df.select(
        F.sum(F.when(parsed.isNull() & F.col(c).isNotNull(), 1).otherwise(0)).alias("invalid_dates"),
        F.min(parsed).alias("min_date"),
        F.max(parsed).alias("max_date")
    ).show()


=== Checking order_date (Correct format: yyyy-MM-dd) ===
+-------------+-------------------+-------------------+
|invalid_dates|           min_date|           max_date|
+-------------+-------------------+-------------------+
|            0|2025-01-01 00:00:00|2026-09-30 00:00:00|
+-------------+-------------------+-------------------+



In [14]:
print("=== Removing Duplicate Rows ===")
df = df.dropDuplicates()

=== Removing Duplicate Rows ===


In [15]:
print("=== Handling Missing Values ===")

text_cols = [c for c, t in df.dtypes if t == "string"]
num_cols = [c for c, t in df.dtypes if t in ["int", "bigint", "float", "double"]]

# Text columns
df = df.fillna("Unknown", subset=text_cols)

# Numerical columns
for c in num_cols:
    median = df.approxQuantile(c, [0.5], 0.01)[0]
    df = df.fillna({c: median})

=== Handling Missing Values ===


In [16]:
print("=== Cleaning Text Columns ===")

for c in text_cols:
    df = df.withColumn(c, F.trim(F.col(c)))
if "category" in df.columns:
    df = df.withColumn("category", F.initcap(F.col("category")))

=== Cleaning Text Columns ===


In [17]:
print("=== Validating Numerical Values ===")
from pyspark.sql import functions as F

df = df.withColumn(
    "quantity",
    F.when(F.col("quantity") > 0, F.col("quantity"))
)

df = df.withColumn(
    "unit_price",
    F.when(F.col("unit_price") >= 0, F.col("unit_price"))
)

df = df.withColumn(
    "discount_percent",
    F.when(
        F.col("discount_percent").between(0, 100),
        F.col("discount_percent")
    )
)

=== Validating Numerical Values ===


In [18]:
print("=== Cleaning Date Columns ===")

date_cols = [c for c in df.columns if "date" in c.lower()]

for c in date_cols:
    df = df.withColumn(
        c,
        F.expr(f"try_cast(`{c}` as date)")
    )

=== Cleaning Date Columns ===


In [19]:
print("=== Post-Validation Diagnostic Check ===")
from pyspark.sql import functions as F

# Let's count how many null values exist in each column now
null_diagnostics = [
    F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in df.columns
]

print("Current Null counts across columns:")
df.select(null_diagnostics).show()

print("Summary of numerical ranges:")
for col_name in ["quantity", "unit_price", "discount_percent"]:
    df.select(
        F.min(col_name).alias(f"min_{col_name}"),
        F.max(col_name).alias(f"max_{col_name}")
    ).show()

=== Post-Validation Diagnostic Check ===
Current Null counts across columns:
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|order_id|customer_id|customer_name|product|category|quantity|unit_price|discount_percent|payment_method|city|state|order_date|order_status|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+
|       0|          0|            0|      0|       0|       0|         0|               0|             0|   0|    0|         6|           0|
+--------+-----------+-------------+-------+--------+--------+----------+----------------+--------------+----+-----+----------+------------+

Summary of numerical ranges:
+------------+------------+
|min_quantity|max_quantity|
+------------+------------+
|           1|          10|
+------------+------------+

+--------------+--------------+
|min_unit_price

In [20]:
print("=== Handling Numerical Nulls ===")

num_cols = ["quantity", "unit_price", "discount_percent"]

for c in num_cols:
    median = df.approxQuantile(c, [0.5], 0.01)[0]
    df = df.fillna({c: median})

=== Handling Numerical Nulls ===


In [21]:
print("=== Date Nulls Preserved ===")

df.filter(F.col("order_date").isNull()).show()

=== Date Nulls Preserved ===
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|order_id|customer_id|customer_name|      product| category|quantity|unit_price|discount_percent|  payment_method|     city|        state|order_date|order_status|
+--------+-----------+-------------+-------------+---------+--------+----------+----------------+----------------+---------+-------------+----------+------------+
|  100634|      C0262|  Tanvi Mehta|    Atta 10kg|  Grocery|      10|       560|              15|Cash on Delivery|Hyderabad|    Telangana|      NULL|  Processing|
|  100969|      C0277| Swathi Patel| Sports Shoes| Clothing|       4|      6000|              20|             UPI|Hyderabad|    Telangana|      NULL|   Delivered|
|  100358|      C0132| Vikram Menon|Bedside Table|Furniture|       7|      4460|              15|     Net Banking|  Lucknow|Uttar Pradesh|      NULL|   Deli

In [22]:
import shutil
import os

silver_path = "data_lake/silver/preprocessed_sales"
temp_csv_path = "data_lake/silver/preprocessed_sales_temp"

print("=== Saving Cleaned Dataset as CSV directly to preprocessed_sales ===")

df.coalesce(1).write.mode("overwrite").option("header", "true").csv(temp_csv_path)

if os.path.exists(silver_path):
    shutil.rmtree(silver_path)

os.rename(temp_csv_path, silver_path)
print(f"Successfully saved cleaned CSV directly to: {silver_path}")

print("\n=== Contents of the Silver Zone Directory ===")
for root, dirs, files in os.walk('data_lake/silver'):
    level = root.replace('data_lake/silver', '').count(os.sep)
    indent = ' ' * 4 * level
    print(f"{indent}{os.path.basename(root)}/")
    for f in files:
        if not f.startswith('.'):
            print(f"{indent}    {f}")

=== Saving Cleaned Dataset as CSV directly to preprocessed_sales ===
Successfully saved cleaned CSV directly to: data_lake/silver/preprocessed_sales

=== Contents of the Silver Zone Directory ===
silver/
    preprocessed_sales/
        part-00000-2ee86476-6d93-4c25-94f1-058ed4a4913f-c000.csv
        _SUCCESS


In [23]:
print("=== Reading Preprocessed Silver Data (CSV) ===")
silver_path = "data_lake/silver/preprocessed_sales"
silver_df = spark.read.csv(silver_path, header=True, inferSchema=True)

print(f"Silver Dataset Record Count: {silver_df.count()}")
print("\n=== Schema ===")
silver_df.printSchema()

print("\n=== Sample Records ===")
silver_df.show(5, truncate=False)

=== Reading Preprocessed Silver Data (CSV) ===
Silver Dataset Record Count: 990

=== Schema ===
root
 |-- order_id: integer (nullable = true)
 |-- customer_id: string (nullable = true)
 |-- customer_name: string (nullable = true)
 |-- product: string (nullable = true)
 |-- category: string (nullable = true)
 |-- quantity: integer (nullable = true)
 |-- unit_price: integer (nullable = true)
 |-- discount_percent: integer (nullable = true)
 |-- payment_method: string (nullable = true)
 |-- city: string (nullable = true)
 |-- state: string (nullable = true)
 |-- order_date: date (nullable = true)
 |-- order_status: string (nullable = true)


=== Sample Records ===
+--------+-----------+----------------+-------------------+-----------+--------+----------+----------------+----------------+-------+-----------+----------+------------+
|order_id|customer_id|customer_name   |product            |category   |quantity|unit_price|discount_percent|payment_method  |city   |state      |order_date|orde